<a href="https://colab.research.google.com/github/brahimia/.github.io/blob/main/Demographic-Aware%20RAG%20for%20AI%20Policy%20Compliance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving NIST.AI.100-1.pdf to NIST.AI.100-1.pdf


In [2]:
uploaded = files.upload()

Saving NIST.CSWP.40.ipd.pdf to NIST.CSWP.40.ipd.pdf


In [3]:
!pip install -q pymupdf sentence-transformers tiktoken openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 58.6 MB/s eta 0:00:00


In [4]:
import fitz
import os

documents = []

for filename in uploaded.keys():

    pdf = fitz.open(filename)

    for page_number, page in enumerate(pdf):

        text = page.get_text()

        if text.strip():

            documents.append({
                "document": filename,
                "page": page_number + 1,
                "text": text
            })

print("Pages extracted:", len(documents))

Pages extracted: 51


In [5]:
documents[0]

{'document': 'NIST.CSWP.40.ipd.pdf',
 'page': 1,
 'text': 'NIST Cybersecurity White Paper \n1 \n2 \n3 \n4 \n5 \n6 \n7 \n8 \n9 \nCSWP 40 ipd (Initial Public Draft) \nNIST Privacy Framework 1.1 \nInitial Public Draft\nNational Institute of Standards and Technology \nThis publication is available free of charge from: https://doi.org/10.6028/NIST.CSWP.40.ipd \nApril 14, 2025\n'}

In [6]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=400, overlap=80):

    tokens = encoding.encode(text)

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + chunk_size

        chunk_tokens = tokens[start:end]

        chunk = encoding.decode(chunk_tokens)

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [7]:
chunks = []

chunk_id = 0

for doc in documents:

    page_chunks = chunk_text(doc["text"])

    for text in page_chunks:

        chunks.append({
            "chunk_id": chunk_id,
            "document": doc["document"],
            "page": doc["page"],
            "text": text
        })

        chunk_id += 1

print("Total chunks:", len(chunks))

Total chunks: 112


In [8]:
demographic_groups = {

    "race_ethnicity":
    """
    Race, ethnicity, national origin, racial discrimination,
    ethnic discrimination, disparate impact, racial bias,
    protected racial or ethnic groups and discrimination
    involving AI systems.
    """,

    "gender_sex":
    """
    Sex, gender, women, men, gender identity,
    sex discrimination, gender discrimination,
    gender bias and unequal treatment involving AI systems.
    """,

    "disability":
    """
    Disability, disabled persons, accessibility,
    reasonable accommodation, assistive technology,
    discrimination based on disability and unequal
    performance of AI systems affecting people with disabilities.
    """,

    "age":
    """
    Age discrimination, older adults, elderly people,
    younger people and unequal treatment or algorithmic
    outcomes associated with age.
    """,

    "national_origin":
    """
    National origin, immigration background, nationality,
    language, accent and discrimination involving
    national origin or linguistic characteristics.
    """,

    "religion":
    """
    Religion, religious beliefs, religious practices,
    religious discrimination and AI systems affecting
    religious groups.
    """
}

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
group_names = list(demographic_groups.keys())

group_descriptions = [
    demographic_groups[group]
    for group in group_names
]

group_embeddings = model.encode(
    group_descriptions,
    normalize_embeddings=True
)

In [14]:
similarity = chunk_embeddings @ group_embeddings.T

In [15]:
import numpy as np

similarity_matrix = np.dot(
    chunk_embeddings,
    group_embeddings.T
)

similarity_matrix.shape

(112, 6)

In [16]:
THRESHOLD = 0.35

demographic_collections = {
    group: []
    for group in group_names
}

for i, chunk in enumerate(chunks):

    for j, group in enumerate(group_names):

        score = similarity_matrix[i, j]

        if score >= THRESHOLD:

            demographic_collections[group].append({

                **chunk,

                "classification_score": float(score),

                "embedding": chunk_embeddings[i]
            })

In [17]:
demographic_collections["disability"]

[{'chunk_id': 25,
  'document': 'NIST.CSWP.40.ipd.pdf',
  'page': 13,
  'text': 'CSWP 40 ipd (Initial Public Draft) \n \nNIST Privacy Framework 1.1 \nApril 14, 2025 \n \n   \n7 \n \n \n \n304 \n305 \n306 \n307 \n308 \n309 \n310 \n311 \n312 \n313 \n314 \n315 \n316 \n317 \n318 \n319 \n320 \n321 \n322 \n323 \n324 \n325 \n \n1.2.2. Artificial Intelligence and Privacy Risk Management \nArtificial intelligence (AI) systems are engineered or machine-based systems that can, for a \ngiven set of objectives, generate outputs such as predictions, recommendations, or decisions \ninfluencing real or virtual environments. As a tool designed for all technologies, Privacy \nFramework 1.1 can assist organizations with identifying and managing privacy risks that can \narise from data processing within AI systems throughout the AI lifecycle. Privacy risks can arise, \nfor example, when AI systems are trained on data that was collected without individuals’ \nconsent or have missing or inadequate privacy s

In [18]:
sorted_chunks = sorted(
    demographic_collections["disability"],
    key=lambda x: x["classification_score"],
    reverse=True
)

for chunk in sorted_chunks[:5]:

    print("\n-------------------------")
    print("Score:", round(chunk["classification_score"], 3))
    print("Source:", chunk["document"])
    print("Page:", chunk["page"])
    print(chunk["text"][:800])


-------------------------
Score: 0.397
Source: NIST.CSWP.40.ipd.pdf
Page: 13
 Managing Bias in Artificial Intelligence. (National Institute of Standards and Technology, Gaithersburg, MD), 
NIST Special Publication 1270. Available at https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.1270.pdf  
Figure 3: Relationship Between Privacy Risk and Enterprise Risk 


-------------------------
Score: 0.386
Source: NIST.CSWP.40.ipd.pdf
Page: 14
 
the information that can help organizations to weigh the benefits of the data processing against 
the risks and to determine the appropriate response—sometimes referred to as 
 
13  
See NIST Artificial Intelligence (AI) Risk Management Framework (AI RMF 1.0), NIST AI 100-1 at [6].  
14  
See, e.g., NIST Data Governance and Management Profile. Available at https://www.nist.gov/privacy-framework/new-projects/data-
governance-and-management-profile. 


-------------------------
Score: 0.381
Source: NIST.CSWP.40.ipd.pdf
Page: 13
CSWP 40 ipd (In

In [19]:
sorted_chunks = sorted(
    demographic_collections["race_ethnicity"],
    key=lambda x: x["classification_score"],
    reverse=True
)

for chunk in sorted_chunks[:5]:

    print("\n-------------------------")
    print("Score:", round(chunk["classification_score"], 3))
    print(chunk["text"][:800])


-------------------------
Score: 0.547
create privacy-invasive images, video, or audio). These and other data processing activities 
within AI systems may create privacy problems for individuals and groups, including at a 
societal level, ranging from dignity effects to more concrete harms like physical harm and 
economic loss. As discussed in Section 1.2.1 above, these impacts on the privacy of individuals 
and groups can lead to significant organizational impacts, ranging from revenue losses to 
reputational harms. 
 
11  
Numerous publications analyze and characterize AI privacy risks. See, for example, Lee H, Yang Y, von Davier TS, Forlizzi J, Das S (2024) 
Deepfakes, Phrenology, Surveillance, and More! A Taxonomy of AI Privacy Risks. (Carnegie Mellon University, Pittsburgh PA, United 
States). Available at https://dl.acm.o

-------------------------
Score: 0.534
 Managing Bias in Artificial Intelligence. (National Institute of Standards and Technology, Gaithersburg, MD), 
NIST Sp

In [20]:
def retrieve(query, group, top_k=5):

    collection = demographic_collections[group]

    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    results = []

    for item in collection:

        score = np.dot(
            query_embedding,
            item["embedding"]
        )

        results.append({
            **item,
            "retrieval_score": float(score)
        })

    results = sorted(
        results,
        key=lambda x: x["retrieval_score"],
        reverse=True
    )

    return results[:top_k]

In [21]:
query = """
An AI video analytics system used for public safety
produces more false positive anomaly alerts for
people using wheelchairs or mobility aids.
What governance and discrimination issues should
the organization consider?
"""

results = retrieve(
    query,
    group="disability",
    top_k=5
)

In [22]:
for r in results:

    print("\n=========================")

    print(
        r["document"],
        "| Page:",
        r["page"],
        "| Similarity:",
        round(r["retrieval_score"], 3)
    )

    print(r["text"][:1000])


NIST.CSWP.40.ipd.pdf | Page: 13 | Similarity: 0.543
create privacy-invasive images, video, or audio). These and other data processing activities 
within AI systems may create privacy problems for individuals and groups, including at a 
societal level, ranging from dignity effects to more concrete harms like physical harm and 
economic loss. As discussed in Section 1.2.1 above, these impacts on the privacy of individuals 
and groups can lead to significant organizational impacts, ranging from revenue losses to 
reputational harms. 
 
11  
Numerous publications analyze and characterize AI privacy risks. See, for example, Lee H, Yang Y, von Davier TS, Forlizzi J, Das S (2024) 
Deepfakes, Phrenology, Surveillance, and More! A Taxonomy of AI Privacy Risks. (Carnegie Mellon University, Pittsburgh PA, United 
States). Available at https://dl.acm.org/doi/pdf/10.1145/3613904.3642116; and Solove DJ (2024) Artificial Intelligence and Privacy. 77 
Florida Law Review, GWU Legal Studies Research Pa